# Inventory Reconciliation Engine
### My C programming journal — from a messy first attempt to a cleaner design

**Alireza Jamshidian Tehrani**  
**October 2026** · C programming · File parsing · Dynamic memory · Debugging

> I wanted this to be a small command-line project that would force me to think like a programmer, not just write code that looks right. It ended up being more difficult than I expected, mainly because of decisions I made before I even started writing the real logic.

This notebook is my record of that process: what I tried, where I got stuck, why I decided to start again, the changes I made, and what I understand better now. It is a development journal, **not a claim that every test or edge case has already passed**.

## 01 — What I was trying to build

The idea is straightforward. A company has a file showing how many units of every product should be in stock, and another file with the actual physical counts. My program has to compare them and report what is wrong.

**`expected.txt`**

```text
P1001|Keyboard|50
P1002|Mouse|100
P1003|Monitor|25
```

**`counted.txt`**

```text
P1001|45
P1002|110
P1003|25
X9999|8
BROKEN LINE
```

From these two files I need to distinguish **shortage**, **surplus**, and **exact match**. I also need to count **unknown product IDs** and **malformed records** without allowing one bad line to crash the whole program.

The catch is that I don't know in advance how many products are going to appear. I cannot solve the problem properly by pretending the input will always be small, clean, and predictable.

That is exactly why I picked this exercise. I wanted to practise C as a language where I am responsible for the memory, the input, and the consequences of my own design.

## 02 — My first mistake: coding before the design was clear

My first approach was to define some structures, create parsing functions, and just get the files loaded. I had separate `Product` and `Inventory` ideas, pointer fields for product names and IDs, and different functions that were supposed to read and store data.

On paper I thought I knew what every piece should do. In the code, the responsibilities started overlapping.

Some of the problems I ran into:

| What I was doing | Why it became a problem |
|---|---|
| Using `char *` members before properly allocating storage | A pointer does not automatically give me a writable string buffer. |
| Moving between fixed-size arrays and pointers | I was avoiding the real problem of handling an unknown number of products. |
| Mixing product information with reconciliation information | I was not sure which structure owned which values. |
| Trying to parse directly into my final stored objects | One malformed line could leave partly updated data behind. |
| Writing functions before deciding their return values | Errors were difficult to report back to the caller. |
| Using `atoi()` for quantities | I could not reliably distinguish bad input from a legitimate zero. |

The issue wasn't that one line had a typo. I had several basic decisions that didn't agree with each other. Fixing one function sometimes created another problem somewhere else.

**What I learned here:** before implementing a parser, I need a clear answer to three questions: *Where does the data live? Who owns it? What does a function promise to return?*

## 03 — Why I decided to start again

At some point, debugging stopped being useful. I was changing pointer types, switching between arrays and dynamically allocated memory, and adjusting parsing functions without having a stable model of the program.

I also made mistakes with values and pointers. A function receiving a copy of a count does not update the caller's count. A pointer that has not been given memory cannot safely be treated like an array. These are basic details in C, but when they happen inside a bigger function they are easy to overlook.

My restart wasn't about throwing away the whole project. It was about **throwing away a confusing structure before I built more code on top of it**.

I reduced the program to three responsibilities:

1. **Store** the products somewhere safe, even as the list grows.
2. **Parse** each line completely and reject it if it is invalid.
3. **Reconcile** the quantities and produce the report.

That was the first real improvement: instead of thinking *"How do I finish this function?"*, I started asking *"What is this function allowed to change?"*

## 04 — The new data model

The version I moved toward was much simpler: one `Product` record, one growing `ProductList`, and one `Report` for statistics. The sketch below shows the design idea; the source files in my repository remain the reference for exact declarations.

```c
typedef enum {
    SURPLUS,
    SHORTAGE,
    MATCH,
    MALFORMED
} InventoryStatus;

typedef struct {
    char id[20];
    char name[20];
    size_t counted;
    size_t expected;
    InventoryStatus status;
} Product;

typedef struct {
    Product *item;
    size_t count;
    size_t capacity;
} ProductList;
```

The key decision was `ProductList`. I stopped expecting the list to somehow manage itself. I needed to keep track of **three different things**:

- `item` — the memory address of my allocated product array.
- `count` — how many valid products are currently stored.
- `capacity` — how many products the allocated block can hold before it must grow.

I also kept statistics in a separate `Report` structure instead of spreading counters everywhere.

This made the program easier to reason about. The parser shouldn't have to invent its own storage rules, and the storage function shouldn't have to know the meaning of a shortage.

## 05 — `append()`: the part that changed how I understand memory

This was one of the most useful parts of the project for me.

At first, adding a product sounded easy: find the next empty position and copy it there. But that assumes an empty position exists. If the number of products is unknown, the list has to grow when it becomes full.

I ended up working through the logic of an append operation:

```text
Before adding a product:

item  ──► [ Product ][ Product ][ empty ][ empty ]
           count = 2                capacity = 4

If count == capacity:
    calculate a larger capacity
    request a larger memory block with realloc()
    only update the list if realloc succeeds

Then:
    item[count] = new_product
    count++
```

The important distinction is that **`count` and `capacity` are not the same thing**. Capacity describes reserved space. Count describes objects I actually have.

I started with capacity **1**, then used doubling to grow the list instead of reallocating every time I added one more product.

### The `realloc()` mistake I learned to avoid

One of the most important corrections was not assigning the result of `realloc()` directly to the only pointer I have.

**Risky:**

```c
list->item = realloc(list->item, new_capacity * sizeof *list->item);
```

If that call fails, `realloc()` returns `NULL`. By overwriting `list->item`, I would lose my original allocation and have no way to free or continue using it.

**Safer pattern (simplified excerpt):**

```c
Product *temporary = realloc(
    list->item,
    new_capacity * sizeof *list->item
);

if (temporary == NULL) {
    return -1;                 // old allocation is still valid
}

list->item = temporary;
list->capacity = new_capacity;
```

I also had to think about the less obvious problem: **overflow**. Before doubling the capacity or multiplying by `sizeof(Product)`, I should make sure the calculation actually fits into `size_t`.

This was the point where dynamic memory stopped being just `malloc()`/`realloc()` syntax. I had to think about failure, ownership, and keeping the structure consistent even when an operation doesn't work.

### A stripped-down example of the append logic

This example demonstrates the approach I was learning. It is here to explain the idea, not to replace the implementation in `inventory.c`.

```c
#include <stdint.h>  // SIZE_MAX
#include <stdlib.h>  // realloc

int list_append(ProductList *list, const Product *product) {
    if (list == NULL || product == NULL) return -1;

    if (list->count == list->capacity) {
        size_t next_capacity;

        if (list->capacity == 0) {
            next_capacity = 1;
        } else {
            if (list->capacity > SIZE_MAX / 2) return -1;
            next_capacity = list->capacity * 2;
        }

        if (next_capacity > SIZE_MAX / sizeof *list->item) return -1;

        Product *temporary = realloc(
            list->item,
            next_capacity * sizeof *list->item
        );
        if (temporary == NULL) return -1;

        list->item = temporary;
        list->capacity = next_capacity;
    }

    list->item[list->count] = *product;
    list->count++;
    return 0;
}
```

I used to think the main job was making the append succeed. Now I see that a good append function must also **leave the list usable when it fails**.

## 06 — Parsing: getting text into memory isn't the same as validating it

Parsing was another place where I kept running into problems.

The input looks simple because each line has a `|` delimiter. I tried using `sscanf()` to split the fields, but there were several things I had to pay attention to:

- A `%[...]` conversion writes text into a character buffer. It cannot be used as though the destination were an integer.
- Field widths matter. A string that is longer than my buffer must not overflow it.
- The number of successful conversions matters. A broken line may contain only some of the fields.
- Even if I successfully read a quantity as text, that does **not** prove it is a valid number.
- Trailing garbage, negative values, and values outside the numeric range need explicit checks.

I moved away from relying on `atoi()` and worked with `strtol()`, `errno`, and an `endptr` so I could check what really happened during numeric conversion.

For example, the idea behind my validation became:

```c
errno = 0;
char *endptr;
long value = strtol(quantity_text, &endptr, 10);

if (errno == ERANGE ||
    endptr == quantity_text ||
    *endptr != '\0' ||
    value < 0) {
    // Reject the quantity as invalid.
}
```

Before storing a nonnegative value into a `size_t`, I still have to check that it fits the destination type. And if the input has trailing spaces, I have to decide whether to trim them or reject them consistently.

The biggest lesson: **valid data is something I establish, not something I assume because a library function returned a value**.

## 07 — Why temporary variables made debugging easier

I learned that there is a big difference between these two workflows:

```text
Bad direction:
Read line → partially overwrite a stored Product → discover an error

Better direction:
Read line → parse into temporary fields → validate everything
          → find or append the Product → update/report
```

When I try to write into the final product too early, I have to undo the changes if the line turns out to be invalid. That is unnecessary complexity.

Creating a fresh temporary product or temporary fields for each line gave me a more predictable flow. The object either passes validation and can be stored, or it is rejected and the permanent list stays unchanged.

The same applies to file errors. `fopen()` can fail. Allocation can fail. Parsing can fail. Those are different kinds of failure, and I should not treat them all as the same thing.

I also understood why a parser needs a meaningful return value instead of always being `void`: the caller needs to know whether it can continue, stop, or clean up.

## 08 — Searching for an ID and doing the actual reconciliation

Once the list and the parser were more understandable, I could separate another issue: **finding a product is not the same as counting its stock**.

The intended order is:

1. Read expected inventory and store each known product once.
2. Read counted inventory line by line.
3. For each valid counted record, search for the ID in the expected list.
4. If there is no matching product, track it as an unknown ID rather than inventing a new expected product.
5. If it exists, update the count and calculate the difference.

The rule is simple:

```text
difference = counted - expected

negative → SHORTAGE
zero     → MATCH
positive → SURPLUS
```

But I had to be careful with **unsigned types**. My quantities can be `size_t`, but subtracting a larger expected quantity from a smaller counted quantity using unsigned arithmetic can wrap around. Comparing the quantities first is safer than casually storing a negative result in `size_t`.

Another thing I needed to question: what happens if a product exists in the expected file but **never appears** in the counted file? A default count of zero could mean a real count of zero, or it could mean that the item was never counted. I shouldn't call it an exact match by accident.

And the summary statistics should describe products accurately: if the same product is processed more than once, I don't want to blindly increase its shortage or surplus count twice.

## 09 — My debugging process, in a more organized form

Looking back, I can group most of my mistakes into a small number of categories.

| What went wrong or looked suspicious | What I started checking instead |
|---|---|
| The parser failed on odd lines | What exactly did `sscanf()` read, and did I validate every field? |
| `atoi()` gave me an integer anyway | Can `strtol()` show invalid characters, range errors, and negative input? |
| A pointer existed but had no storage | Where was the memory allocated, and how long does it remain valid? |
| The list grew inconsistently | Does `append()` maintain `count <= capacity` after every path? |
| Allocation could fail | Am I retaining the old pointer if `realloc()` returns `NULL`? |
| Values in functions didn't update as expected | Did I pass an object by value instead of passing its address? |
| My report numbers might be misleading | Am I counting records or unique products, and at what point? |
| I included `inventory.c` in `main.c` | Am I confusing source inclusion with compiling and linking modules? |

I didn't solve these by writing more lines. I made progress when I isolated one assumption at a time, compared it with what C actually does, and adjusted the design rather than patching symptoms everywhere.

I also saw the value of compiler warnings. Compiling with `-Wall -Wextra -std=c11` is a cheap way to catch problems before I waste time looking for them at runtime.

## 10 — Testing: I need cases that try to break my assumptions

Initially my instinct was to make a normal file, run the code, and decide that it worked if it produced numbers. That is not enough for a parser.

The test cases I need are not just successful examples. They are situations where my assumptions fail.

| Test case | What the test is trying to prove |
|---|---|
| A normal match | Basic matching and report logic |
| Expected quantity larger than counted | Shortage is classified correctly |
| Counted quantity larger than expected | Surplus is classified correctly |
| An unknown product ID | Unknown records do not silently become products |
| `BROKEN LINE` | Malformed input is counted without a crash |
| A negative quantity such as `-2` | Invalid physical quantities are rejected |
| `12abc` as a quantity | Numeric conversion checks trailing garbage |
| An extremely large quantity | Overflow and range checking work |
| An empty input file | Empty inputs do not access nonexistent elements |
| More products than the initial capacity | `realloc()` and append really work |
| Duplicate IDs | The program follows a clear duplicate policy |
| An expected product missing from counted data | The result is deliberate, not an accidental match |

These are **planned checks**, not claims about tests I have already passed. The point is to stop depending on one happy-path run.

### A small reproducible fixture

The following Python cell creates **separate test files** in a `notebook-fixtures/` folder. It won't overwrite the normal `expected.txt` or `counted.txt` in the root of my project. I use Python here only as a convenient notebook tool for preparing inputs; the application itself remains C.

In this fixture, **if my intended validation and reconciliation rules are implemented correctly**, I expect: 1 shortage, 1 surplus, 1 exact match, 1 unknown ID, and 2 malformed lines. This is an *expected result*, not a recorded successful run.

In [ ]:
from pathlib import Path

project = Path.cwd()
fixture_dir = project / "notebook-fixtures"
fixture_dir.mkdir(exist_ok=True)

(fixture_dir / "expected.txt").write_text(
    "A101|Filter|12\n"
    "B205|Valve|7\n"
    "C309|Pump|0\n",
    encoding="utf-8",
)

(fixture_dir / "counted.txt").write_text(
    "A101|10\n"
    "B205|7\n"
    "C309|3\n"
    "X999|2\n"
    "BROKEN LINE\n"
    "A101|-2\n",
    encoding="utf-8",
)

print("Fixture files created in:", fixture_dir)
print("Expected summary (to verify, not assumed):")
print("shortage=1, surplus=1, exact=1, unknown=1, malformed=2")

### Compile and run the actual C code (optional)

When this notebook is in the same folder as `main.c` and `inventory.c`, the next cell tries to compile the real project with warnings enabled, then runs it **from the fixture directory** so relative file reads can pick up the sample data.

This only works if my current parsing functions read `expected.txt` and `counted.txt` relative to the working directory. If I later add command-line arguments, I will need to adjust the run command. Compile errors and runtime output are shown rather than hidden.

In [ ]:
import os
import subprocess

sources = [project / "main.c", project / "inventory.c"]
if not all(source.is_file() for source in sources):
    print("Put this notebook alongside main.c and inventory.c to compile the project.")
else:
    executable = fixture_dir / ("inventory-check.exe" if os.name == "nt" else "inventory-check")
    command = [
        "gcc", "-std=c11", "-Wall", "-Wextra",
        str(sources[0]), str(sources[1]), "-o", str(executable)
    ]
    try:
        build = subprocess.run(command, capture_output=True, text=True, timeout=30)
        print("Compiler return code:", build.returncode)
        if build.stdout:
            print("Compiler stdout:\n", build.stdout)
        if build.stderr:
            print("Compiler messages:\n", build.stderr)
        if build.returncode == 0:
            result = subprocess.run(
                [str(executable)], cwd=fixture_dir,
                capture_output=True, text=True, timeout=10
            )
            print("Program return code:", result.returncode)
            print("Program stdout:\n", result.stdout)
            if result.stderr:
                print("Program stderr:\n", result.stderr)
    except FileNotFoundError:
        print("GCC is not on PATH; install/configure it before using this cell.")
    except subprocess.TimeoutExpired:
        print("The compilation or program exceeded the timeout; investigate possible blocking/looping.")

## 11 — The `main.c` milestone

After spending most of my time on memory handling and parsing, I finally reached the stage where I could move into `main.c` and let the functions work together.

My first `main.c` initialized the list and report, called both parsers, and printed the statistics. That was progress because I had finally separated the orchestration from the lower-level functions.

I still made a classic C project mistake: I included both `inventory.h` and `inventory.c` inside `main.c`.

```c
#include "inventory.h"
#include "inventory.c"  // This shouldn't be included here.
```

I corrected the idea: headers declare the shared interface; `.c` files should normally be compiled as separate translation units and linked together.

```bash
gcc -Wall -Wextra -std=c11 main.c inventory.c -o inventory-check
```

The other missing piece in my first main was **cleanup**. If I allocate an array during loading, I need to free it before exit, including on error paths where applicable.

At this point, the high-level flow makes sense to me:

```text
Initialize list and report
        ↓
Load expected inventory
        ↓
Process counted inventory
        ↓
Print reconciliation summary
        ↓
Release dynamically allocated memory
```

Getting to a small `main.c` was not a shortcut. It took a lot of restructuring to make the top-level code this simple.

## 12 — What I actually learned from the restart

There are a few things I understand differently now.

**1. A working data structure is part of the solution.**  
I used to treat storage as a detail I could fix while writing the parser. Now I understand why deciding ownership, capacity, and lifetime first saves work later.

**2. Dynamic memory comes with responsibilities.**  
`realloc()` isn't just a way to make an array bigger. I have to check failure, protect the original pointer, calculate sizes safely, and eventually free the memory.

**3. Parsing means proving that input is valid.**  
Splitting text with `sscanf()` is only the first step. A number can be missing, negative, too large, or followed by garbage. `strtol()`, `errno`, and `endptr` helped me understand that difference.

**4. I should not modify permanent state until I trust the input.**  
Temporary fields and explicit validation are easier to debug than partial updates that I then have to undo.

**5. Compiler warnings, small test cases, and error paths matter.**  
The question isn't only whether a normal file runs. The question is what happens when a line is wrong or an allocation doesn't work.

**6. Starting again can be the right technical decision.**  
My initial problem wasn't a lack of code. It was that the code didn't have a consistent design. Restarting made me face the fundamentals instead of hiding the mistakes under more functions.

## Final reflection

This project looked small, and that made me underestimate it. I thought most of the work would be comparing two quantities. In reality, the difficult parts were deciding how to represent the data, handling memory without losing it, understanding what my parser was really reading, and keeping errors from corrupting the rest of the program.

There were moments when I was rewriting the same area again and again and not really moving forward. The restart was frustrating, but it made the code easier for me to explain. I can now say why I need `count` and `capacity`, why I use a temporary pointer with `realloc()`, why `atoi()` was not enough, and why a parser should not silently accept broken input.

I am not documenting this as a perfectly clean development story. I made avoidable mistakes, went back to basics, and had to debug things that I thought I already understood. But this is exactly the kind of progress I wanted from these C exercises: not just finishing another assignment, but becoming more careful and more independent in the way I solve programming problems.

**The most useful lesson:** writing the code is only part of the job. Designing it so I can understand, test, and trust it is the harder part.